# COVERX Binary vs Text Parsing Validation

This notebook validates that `read_coverx` produces identical `CovMat` objects
whether reading from a binary or text COVERX file.

In [1]:
import numpy as np
from kika.cov.parse_covmat import read_coverx, _detect_coverx_format, read_scale_covmat

## 1. Format auto-detection

In [2]:
binary_path = r"c:\Users\Usuario\BaradDur\Dev\kika\files\cov\scale.rev05.44groupcov"
text_path   = r"c:\Users\Usuario\BaradDur\Dev\kika\files\cov\scale.rev05.44groupcov.txt"

print(f"Binary file detected as: {_detect_coverx_format(binary_path)}")
print(f"Text file detected as:   {_detect_coverx_format(text_path)}")

Binary file detected as: binary
Text file detected as:   text


## 2. Load both files

In [3]:
cov_bin = read_coverx(binary_path)
print(f"Binary: {cov_bin.num_groups} groups, {cov_bin.num_matrices} matrices")

cov_txt = read_coverx(text_path)
print(f"Text:   {cov_txt.num_groups} groups, {cov_txt.num_matrices} matrices")

Binary: 44 groups, 2207 matrices
Text:   44 groups, 2211 matrices


## 3. Energy grid comparison

In [4]:
eg_txt = np.array(cov_txt.energy_grid)
eg_bin = np.array(cov_bin.energy_grid)

print(f"Energy grids identical: {np.array_equal(eg_txt, eg_bin)}")
print(f"Length: text={len(eg_txt)}, binary={len(eg_bin)}")
print(f"Range: [{eg_bin[0]:.2e}, {eg_bin[-1]:.2e}]")

Energy grids identical: True
Length: text=45, binary=45
Range: [1.00e-11, 2.00e+01]


## 4. Matrix-by-matrix comparison

In [5]:
# Build lookup dictionaries
def build_lookup(cov):
    lookup = {}
    for i in range(cov.num_matrices):
        key = (cov.isotope_rows[i], cov.reaction_rows[i],
               cov.isotope_cols[i], cov.reaction_cols[i])
        lookup[key] = cov.matrices[i]
    return lookup

txt_lookup = build_lookup(cov_txt)
bin_lookup = build_lookup(cov_bin)

common   = set(txt_lookup) & set(bin_lookup)
only_txt = set(txt_lookup) - set(bin_lookup)
only_bin = set(bin_lookup) - set(txt_lookup)

print(f"Common matrices:      {len(common)}")
print(f"Only in text:         {len(only_txt)}")
print(f"Only in binary:       {len(only_bin)}")

if only_txt:
    print("\nMatrices only in text (near-zero due to float64 rounding):")
    for k in sorted(only_txt):
        m = txt_lookup[k]
        print(f"  {k}: sum={m.sum():.2e}, nonzero={np.count_nonzero(m)}")

Common matrices:      2207
Only in text:         4
Only in binary:       0

Matrices only in text (near-zero due to float64 rounding):
  (4009, 2, 4009, 104): sum=-6.04e-09, nonzero=1
  (8016, 4, 8016, 16): sum=-7.82e-09, nonzero=1
  (29065, 4, 29065, 106): sum=-1.64e-09, nonzero=1
  (3004009, 2, 3004009, 104): sum=-6.04e-09, nonzero=1


In [6]:
# Compare all common matrices
max_abs_diffs = []
max_rel_diffs = []

for key in common:
    diff = np.abs(txt_lookup[key] - bin_lookup[key])
    max_abs_diffs.append(diff.max())
    denom = np.maximum(np.abs(txt_lookup[key]), 1e-30)
    max_rel_diffs.append((diff / denom).max())

max_abs_diffs = np.array(max_abs_diffs)
max_rel_diffs = np.array(max_rel_diffs)

print(f"Max absolute difference:  {max_abs_diffs.max():.2e}")
print(f"Mean absolute difference: {max_abs_diffs.mean():.2e}")
print(f"Matrices with diff < 1e-6: {(max_abs_diffs < 1e-6).sum()} / {len(common)}")
print(f"\nAll common matrices match within float32 precision: {max_abs_diffs.max() < 1e-5}")

Max absolute difference:  4.99e-07
Mean absolute difference: 1.39e-08
Matrices with diff < 1e-6: 2207 / 2207

All common matrices match within float32 precision: True


## 5. Backward compatibility check

In [7]:
# Old name should still work and auto-detect binary
cov_compat = read_scale_covmat(binary_path)
print(f"read_scale_covmat on binary file: {cov_compat.num_matrices} matrices")
print(f"Matches read_coverx: {cov_compat.num_matrices == cov_bin.num_matrices}")

read_scale_covmat on binary file: 2207 matrices
Matches read_coverx: True


## 6. Isotope/reaction list comparison

In [8]:
txt_isotopes = set(cov_txt.isotope_rows + cov_txt.isotope_cols)
bin_isotopes = set(cov_bin.isotope_rows + cov_bin.isotope_cols)
print(f"Isotopes in text:   {len(txt_isotopes)}")
print(f"Isotopes in binary: {len(bin_isotopes)}")
print(f"Isotopes match:     {txt_isotopes == bin_isotopes}")

txt_reactions = set(cov_txt.reaction_rows + cov_txt.reaction_cols)
bin_reactions = set(cov_bin.reaction_rows + cov_bin.reaction_cols)
print(f"\nReactions in text:   {sorted(txt_reactions)}")
print(f"Reactions in binary: {sorted(bin_reactions)}")
print(f"Reactions match:     {txt_reactions == bin_reactions}")

Isotopes in text:   407
Isotopes in binary: 407
Isotopes match:     True

Reactions in text:   [2, 4, 16, 18, 22, 28, 102, 103, 104, 105, 106, 107]
Reactions in binary: [2, 4, 16, 18, 22, 28, 102, 103, 104, 105, 106, 107]
Reactions match:     True
